# Spatial GFDN: per-channel fit diagnostics

Run `examples/e8_spatial_gfdn.py` first, then set `output_dir` below. This notebook compares the measured and fitted ambisonic RIR channel by channel.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from loguru import logger

sys.path.insert(0, str(Path('../examples').resolve()))
from e8_spatial_gfdn import load_ambisonic_rir

target_path = Path('../rirs/multi-slope/bathroom1_s0001_mic0_foa.wav').resolve()
# Change this to the directory written by e8_spatial_gfdn.py.
output_dir = Path('../output/spatial_gfdn/bathroom_geq_mrstft_edc_sparsity').resolve()
n_sh = 1
nfft = 96000

target, fs = load_ambisonic_rir(str(target_path), nfft=nfft, n_sh=n_sh, dtype=torch.float32)
fitted_filename = 'ir_optim.wav'  # use 'ir_optim_noisy.wav' only to inspect the noise model
fitted_label = 'Spatial GFDN + noise' if fitted_filename.endswith('_noisy.wav') else 'Spatial GFDN (clean)'
fitted, fitted_fs = sf.read(str((output_dir / fitted_filename).resolve()), always_2d=True)
if fitted_fs != fs:
    raise ValueError(f'Sample-rate mismatch: target={fs}, fitted={fitted_fs}')
fitted = torch.as_tensor(fitted[:nfft], dtype=target.dtype).unsqueeze(0)
if fitted.shape != target.shape:
    raise ValueError(f'Shape mismatch: target={tuple(target.shape)}, fitted={tuple(fitted.shape)}')

n_channels = target.shape[-1]
print(f'Order {n_sh}: {n_channels} SH channels, {target.shape[1] / fs:.2f} s at {fs} Hz')

## Per-channel STFT comparison

Each row is one ambisonic channel. The third column is the fitted-minus-reference magnitude in dB; ideally it is close to zero across the time-frequency region containing energy.

In [ ]:
def stft_db(x, n_fft=2048, hop_length=512, floor_db=-100):
    window = torch.hann_window(n_fft, dtype=x.dtype, device=x.device)
    magnitude = torch.stft(x, n_fft=n_fft, hop_length=hop_length, window=window,
                           return_complex=True).abs()
    return 20 * torch.log10(magnitude.clamp_min(1e-8)).clamp_min(floor_db)

def plot_channel_stfts(reference, prediction, fs, n_fft=2048, hop_length=512):
    reference = reference.squeeze(0)
    prediction = prediction.squeeze(0)
    fig, axes = plt.subplots(reference.shape[-1], 3, figsize=(15, 2.5 * reference.shape[-1]),
                             sharex=True, sharey=True, constrained_layout=True)
    for channel, row in enumerate(axes):
        ref_db = stft_db(reference[:, channel], n_fft, hop_length)
        pred_db = stft_db(prediction[:, channel], n_fft, hop_length)
        # Use one scale for the pair, so a level mismatch is visible.
        upper = max(ref_db.max().item(), pred_db.max().item())
        extent = [0, reference.shape[0] / fs, 0, fs / 2]
        for ax, image, title in zip(row[:2], (ref_db, pred_db), ('Reference', fitted_label)):
            im = ax.imshow(image.cpu(), origin='lower', aspect='auto', extent=extent,
                           cmap='magma', vmin=upper - 80, vmax=upper)
            ax.set_title(f'{title} — SH channel {channel}')
            fig.colorbar(im, ax=ax, label='Magnitude (dB)')
        diff = pred_db - ref_db
        im = row[2].imshow(diff.cpu(), origin='lower', aspect='auto', extent=extent,
                           cmap='coolwarm', vmin=-20, vmax=20)
        row[2].set_title(f'{fitted_label} − reference — SH channel {channel}')
        fig.colorbar(im, ax=row[2], label='Difference (dB)')
        row[0].set_ylabel('Frequency (Hz)')
    for ax in axes[-1]:
        ax.set_xlabel('Time (s)')
    return fig

plot_channel_stfts(target, fitted, fs);

## Per-channel energy-decay curves

The curves use a broadband Schroeder backward integration, normalized independently in each SH channel. The noise-augmented fitted RIR is used, matching the training losses.

In [ ]:
def edc_db(x, floor_db=-100):
    energy = torch.flip(torch.cumsum(torch.flip(x.square(), (0,)), 0), (0,))
    # energy = energy / energy.max().clamp_min(torch.finfo(x.dtype).eps)
    return 10 * torch.log10(energy.clamp_min(10 ** (floor_db / 10)))

def finite_edc_db(x, fs, floor_db=-100, support_threshold_db=-120):
    """Return an EDC only up to the finite, meaningful GFDN support.

    Schroeder integration of a zero-padded IR is constant after its final
    non-zero sample.  That horizontal continuation is not a noise floor, so
    this helper deliberately stops the plotted optimized EDC there.
    """
    peak = x.abs().max().clamp_min(torch.finfo(x.dtype).eps)
    threshold = peak * 10 ** (support_threshold_db / 20)
    support = torch.nonzero(x.abs() > threshold, as_tuple=False).flatten()
    end = int(support[-1]) + 1 if len(support) else 1
    logger.info(f"IR ends at {end / fs * 1e3:.3f} ms")
    return np.arange(end) / fs, edc_db(x[:end], floor_db)

def plot_channel_edcs(reference, prediction, fs, floor_db=-80):
    reference = reference.squeeze(0)
    prediction = prediction.squeeze(0)
    columns = 3
    rows = int(np.ceil(reference.shape[-1] / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(15, 3.8 * rows),
                             sharex=True, sharey=True, constrained_layout=True)
    time = np.arange(reference.shape[0]) / fs
    for channel, ax in enumerate(axes.flat):
        if channel >= reference.shape[-1]:
            ax.set_visible(False)
            continue
        finite_time_ref, finite_edc_ref = finite_edc_db(reference[:, channel], fs, floor_db)
        ax.plot(finite_time_ref, finite_edc_ref.cpu(), label='Reference', lw=1.5)
        finite_time_opt, finite_edc_opt = finite_edc_db(prediction[:, channel], fs, floor_db)
        ax.plot(finite_time_opt, finite_edc_opt.cpu(), label=f'{fitted_label} (finite)', lw=1.2)
        ax.set(title=f'SH channel {channel}', ylim=(floor_db, 2), xlabel='Time (s)', ylabel='EDC (dB)')
        ax.grid(alpha=0.3)
        ax.legend()
    return fig

plot_channel_edcs(target, fitted, fs);

## Learned common-slope RT60 curves

Each curve is one grouped-feedback GEQ. The output GEQs color the SH components, while these feedback GEQs determine the common decay slopes.

In [ ]:
from flamo.auxiliary.eq import eq_freqs

checkpoints_dir = output_dir / 'checkpoints'
checkpoints = list(checkpoints_dir.glob('model_e*.pt'))
if not checkpoints:
    raise FileNotFoundError(f'No checkpoints found in {checkpoints_dir}')
checkpoint_path = max(checkpoints, key=lambda path: int(path.stem.removeprefix('model_e')))
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
rt60 = checkpoint['_Shell__core.feedback_loop.feedback.attenuation.param'].float()
if rt60.ndim != 2:
    raise ValueError('This plot requires a grouped-GEQ attenuation checkpoint.')

center_freq, shelving_freq = eq_freqs(interval=1)
frequencies = torch.cat((torch.tensor([1.0]), shelving_freq[:1],
                         center_freq, shelving_freq[1:])).numpy()
if frequencies.size != rt60.shape[0]:
    raise ValueError(f'Unexpected GEQ band count: {rt60.shape[0]}')

fig, ax = plt.subplots(figsize=(9, 4.5))
for group in range(rt60.shape[1]):
    ax.semilogx(frequencies, rt60[:, group].clamp(0.05, 10.0),
                marker='o', label=f'Feedback group {group}')
ax.set(xlabel='Frequency (Hz)', ylabel='RT60 (s)',
       title=f'Grouped-GEQ RT60s ({checkpoint_path.name})', ylim=(0, 10.2))
ax.grid(True, which='both', alpha=0.3)
ax.legend()

## Late-tail diffuse SCM

Each image is the magnitude of the late-tail, trace-normalized STFT SCM averaged across frequency. An ideal diffuse field has negligible off-diagonal values; the diagonal reflects the AmbiX/SN3D component power distribution.

In [ ]:
def plot_diffuse_scm(rir, title, n_fft=2048, late_start_s=0.08, f_min=125, f_max=8000):
    """Plot the late-tail trace-normalized SCM of one Ambisonic RIR."""
    rir = rir.squeeze(0) if rir.ndim == 3 else rir
    if rir.ndim != 2:
        raise ValueError('rir must have shape (time, channel) or (batch, time, channel)')
    start = int(late_start_s * fs)
    late_rir = rir[start:]
    window = torch.hann_window(n_fft, dtype=rir.dtype, device=rir.device)
    stft = torch.stft(late_rir.T, n_fft=n_fft, hop_length=n_fft // 4,
                      window=window, return_complex=True)
    
    # R[f] = mean_t y[f, t] y[f, t]^H, retaining all SH cross terms.
    scm = torch.einsum('cft,dft->fcd', stft, stft.conj()) / stft.shape[-1]
    trace = scm.diagonal(dim1=-2, dim2=-1).real.sum(dim=-1).clamp_min(1e-12)
    scm = scm / trace[:, None, None]
    
    frequencies = torch.fft.rfftfreq(n_fft, d=1 / fs).to(rir.device)
    band = (frequencies >= f_min) & (frequencies <= min(f_max, fs / 2))
    scm_magnitude = scm[band].abs().mean(dim=0).cpu().numpy()

    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    image = ax.imshow(scm_magnitude, cmap='magma', vmin=0)
    fig.colorbar(image, ax=ax, label='Mean |R / tr(R)|')
    channels = np.arange(rir.shape[-1])
    ax.set(xticks=channels, yticks=channels, xlabel='SH channel', ylabel='SH channel',
           title=f'{title}: late-tail diffuse SCM ({f_min:g}–{f_max:g} Hz)')
    return fig

plot_diffuse_scm(target, 'Reference');
plot_diffuse_scm(fitted, fitted_label);